# Notebook 03 — Concorrência com asyncio.Semaphore

Este notebook explora como controlar concorrência em sistemas de agentes LLM usando `asyncio.Semaphore`.  
Ele é o aprofundamento técnico do módulo `concurrency.py` do projeto.

**O que você vai aprender:**
- A diferença entre concorrência e paralelismo
- Como o event loop do asyncio funciona
- Por que semáforos são essenciais para workloads de LLM
- Como implementar um `AgentSemaphore` com fila e rejeição

## 1. Concorrência vs Paralelismo

**Paralelismo** significa executar múltiplas tarefas *ao mesmo tempo* — literalmente em paralelo, em múltiplos núcleos de CPU.

**Concorrência** significa gerenciar múltiplas tarefas *de forma intercalada* — uma única thread alterna entre elas enquanto alguma está esperando.

```
Paralelismo (multi-core):
  Core 1: [===Task A===]
  Core 2: [===Task B===]
  Core 3: [===Task C===]

Concorrência (single thread / asyncio):
  Thread: [=A=][await][=B=][await][=A=][await][=C=]...
            A espera I/O  B executa  A volta  C executa
```

### Por que asyncio é concorrente, não paralelo?

`asyncio` usa um **event loop** de thread única. Quando uma corrotina executa `await`, ela cede o controle ao event loop, que pode então executar outra corrotina que esteja pronta. Nenhuma CPU extra é necessária.

Isso é perfeito para workloads **I/O-bound** — como chamadas a APIs de LLM, que passam 99% do tempo esperando a rede. O GIL do Python não é um problema aqui porque não há paralelismo de CPU.

```
Event Loop (simplificado):

  while True:
      task = ready_queue.get()   # pega proxima tarefa pronta
      task.run_until_await()     # executa ate proximo await
      if task.is_waiting_io:
          io_monitor.watch(task) # monitora o I/O
      else:
          ready_queue.put(task)  # devolve para a fila
```

In [ ]:
import asyncio
import time
from datetime import datetime

from rich.console import Console
from rich.table import Table

console = Console()


async def task_basic(task_id: int, duration: float) -> dict:
    """Simulates an async task that takes `duration` seconds."""
    start = time.perf_counter()
    start_ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    await asyncio.sleep(duration)
    elapsed = time.perf_counter() - start
    end_ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    return {"id": task_id, "duration": duration, "elapsed": elapsed, "start": start_ts, "end": end_ts}


async def demo_concurrent_gather():
    """Run 5 tasks concurrently — total time should be ~max(durations), not sum."""
    durations = [2.0, 1.5, 3.0, 0.5, 2.5]
    expected_sequential = sum(durations)
    expected_concurrent = max(durations)

    console.print(f"[bold cyan]Executando {len(durations)} tarefas com asyncio.gather()[/bold cyan]")
    console.print(f"  Tempo esperado (sequencial): [red]{expected_sequential:.1f}s[/red]")
    console.print(f"  Tempo esperado (concorrente): [green]{expected_concurrent:.1f}s[/green]\n")

    wall_start = time.perf_counter()
    results = await asyncio.gather(
        *[task_basic(i, d) for i, d in enumerate(durations)]
    )
    wall_elapsed = time.perf_counter() - wall_start

    table = Table(title="Resultado das Tarefas Concorrentes", show_lines=True)
    table.add_column("Task", style="bold")
    table.add_column("Duracao", justify="right")
    table.add_column("Inicio")
    table.add_column("Fim")

    for r in results:
        table.add_row(
            f"Task-{r['id']}",
            f"{r['duration']:.1f}s",
            r["start"],
            r["end"],
        )

    console.print(table)
    console.print(f"\n[bold]Tempo total real:[/bold] [green]{wall_elapsed:.2f}s[/green]")
    console.print(f"  Ratio vs sequencial: [yellow]{expected_sequential / wall_elapsed:.1f}x mais rapido[/yellow]")


await demo_concurrent_gather()

## 2. O problema sem controle

`asyncio.gather()` sem limitacao dispara **todas as tarefas ao mesmo tempo**. Para 5 tarefas isso pode ser aceitavel. Mas o que acontece quando o sistema recebe 20 requisicoes simultaneas para um agente LLM?

Problemas que surgem:
- **Rate limiting**: a API do LLM rejeita requests acima de um limite (RPM/TPM)
- **Custo explosivo**: 20 chamadas simultaneas = 20x o custo por segundo
- **Esgotamento de conexoes**: connection pools tem tamanho finito
- **Degradacao de qualidade**: o servidor do LLM throttle requests, aumentando latencia

O codigo abaixo demonstra o problema: todas as 20 tarefas iniciam ao mesmo tempo.

In [ ]:
import asyncio
import random
import time
from datetime import datetime

from rich.console import Console
from rich.table import Table

console = Console()

REFERENCE_START: float = 0.0


async def fake_llm_call(task_id: int) -> dict:
    """Simulates an LLM API call with random duration."""
    duration = random.uniform(1.0, 3.0)
    start_offset = time.perf_counter() - REFERENCE_START
    await asyncio.sleep(duration)
    end_offset = time.perf_counter() - REFERENCE_START
    return {
        "id": task_id,
        "duration": duration,
        "start_offset": start_offset,
        "end_offset": end_offset,
    }


async def demo_no_control(n_tasks: int = 20):
    """Launch n_tasks fake LLM calls simultaneously with no rate control."""
    global REFERENCE_START
    REFERENCE_START = time.perf_counter()

    console.print(f"[bold red]Sem controle: disparando {n_tasks} chamadas simultaneas[/bold red]\n")

    results = await asyncio.gather(*[fake_llm_call(i) for i in range(n_tasks)])

    table = Table(title=f"Chamadas sem Semaforo ({n_tasks} tasks)", show_lines=False)
    table.add_column("Task", style="bold", width=8)
    table.add_column("Inicio (s)", justify="right", width=12)
    table.add_column("Fim (s)", justify="right", width=10)
    table.add_column("Duracao", justify="right", width=10)
    table.add_column("Status")

    for r in sorted(results, key=lambda x: x["start_offset"]):
        table.add_row(
            f"Task-{r['id']:02d}",
            f"+{r['start_offset']:.3f}s",
            f"+{r['end_offset']:.3f}s",
            f"{r['duration']:.2f}s",
            "[yellow]ALL STARTED IMMEDIATELY[/yellow]",
        )

    console.print(table)

    starts = [r["start_offset"] for r in results]
    console.print(f"\n[bold]Observacoes:[/bold]")
    console.print(f"  Todas as tasks iniciaram no offset: [red]{min(starts):.3f}s — {max(starts):.3f}s[/red]")
    console.print(f"  Ou seja: [bold red]{n_tasks} chamadas LLM simultaneas![/bold red]")
    console.print(f"  Em producao, isso causaria rate limiting ou custos explosivos.")


await demo_no_control(20)

## 3. Semaforo: a sala de espera

Um **semaforo** e um mecanismo de sincronizacao que controla quantas corrotinas podem executar uma secao critica simultaneamente.

Analogia: imagine uma clinica com apenas 3 consultórios. O recepcionista (semaforo) so deixa entrar 3 pacientes por vez. Os demais ficam na sala de espera ate um consultorio ficar livre.

```
asyncio.Semaphore(3)

  Tasks:  T1  T2  T3  T4  T5  T6  ...
          |   |   |   |   |   |
  [SLOT1] T1------>
  [SLOT2] T2-------->
  [SLOT3] T3------>   T4------>
  [WAIT ]             T5  T6  (esperando slot livre)
```

**API do asyncio.Semaphore:**
```python
sem = asyncio.Semaphore(3)   # maximo 3 corrotinas simultaneas

async with sem:              # acquire (decrementa contador)
    await do_work()          # seção critica
                             # release automatico ao sair (incrementa contador)
```

Quando o contador chega a zero, `await sem.acquire()` bloqueia a corrotina (colocando-a de volta no event loop para esperar) ate que outra libere o semaforo.

In [ ]:
import asyncio
import random
import time
from datetime import datetime

from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

REF_START: float = 0.0
MAX_CONCURRENT = 3


async def fake_llm_call_controlled(task_id: int, semaphore: asyncio.Semaphore) -> dict:
    """LLM call wrapped with semaphore control."""
    global REF_START

    created_offset = time.perf_counter() - REF_START

    async with semaphore:  # blocks here if MAX_CONCURRENT slots are taken
        acquired_offset = time.perf_counter() - REF_START
        wait_time = acquired_offset - created_offset

        duration = random.uniform(1.0, 3.0)
        await asyncio.sleep(duration)

        released_offset = time.perf_counter() - REF_START

    return {
        "id": task_id,
        "created": created_offset,
        "acquired": acquired_offset,
        "released": released_offset,
        "wait_time": wait_time,
        "active_time": duration,
    }


async def demo_with_semaphore(n_tasks: int = 20, max_concurrent: int = MAX_CONCURRENT):
    """Launch n_tasks with semaphore limiting concurrency to max_concurrent."""
    global REF_START
    REF_START = time.perf_counter()

    semaphore = asyncio.Semaphore(max_concurrent)

    console.print(
        f"[bold green]Com Semaforo({max_concurrent}): {n_tasks} tasks, "
        f"maximo {max_concurrent} simultaneas[/bold green]\n"
    )

    results = await asyncio.gather(
        *[fake_llm_call_controlled(i, semaphore) for i in range(n_tasks)]
    )

    table = Table(
        title=f"Chamadas com Semaforo({max_concurrent}) — {n_tasks} tasks",
        show_lines=True,
        box=box.SIMPLE_HEAVY,
    )
    table.add_column("Task", style="bold", width=8)
    table.add_column("Criado (s)", justify="right", width=12)
    table.add_column("Iniciou (s)", justify="right", width=12)
    table.add_column("Terminou (s)", justify="right", width=13)
    table.add_column("Espera", justify="right", width=10)
    table.add_column("Ativo", justify="right", width=10)

    for r in sorted(results, key=lambda x: x["acquired"]):
        waited = r["wait_time"] > 0.05  # small tolerance
        wait_str = f"[red]{r['wait_time']:.2f}s[/red]" if waited else f"[green]{r['wait_time']:.2f}s[/green]"
        table.add_row(
            f"Task-{r['id']:02d}",
            f"+{r['created']:.3f}s",
            f"+{r['acquired']:.3f}s",
            f"+{r['released']:.3f}s",
            wait_str,
            f"{r['active_time']:.2f}s",
        )

    console.print(table)

    waited_tasks = [r for r in results if r["wait_time"] > 0.05]
    console.print(f"\n[bold]Observacoes:[/bold]")
    console.print(f"  Tasks que esperaram na fila: [red]{len(waited_tasks)}/{n_tasks}[/red]")
    console.print(f"  Concorrencia maxima atingida: [green]{max_concurrent}[/green] (nunca ultrapassou)")

    # Verify max concurrency was respected
    # Count how many tasks were active at any given moment (sample at 0.1s intervals)
    total_time = max(r["released"] for r in results)
    max_observed = 0
    for t in [i * 0.1 for i in range(int(total_time / 0.1) + 1)]:
        active = sum(1 for r in results if r["acquired"] <= t < r["released"])
        max_observed = max(max_observed, active)

    console.print(f"  Verificacao — max simultaneas observadas: [bold cyan]{max_observed}[/bold cyan] (limite era {max_concurrent})")


await demo_with_semaphore(20, MAX_CONCURRENT)

## 4. Visualizando o comportamento

A tabela anterior mostra os numeros, mas e difícil ver o padrão de concorrência. Vamos criar uma visualização textual tipo Gantt chart para entender o fluxo de cada task — quanto tempo ela ficou esperando vs quanto tempo ficou ativa.

In [ ]:
import asyncio
import random
import time

from rich.console import Console
from rich.table import Table
from rich import box
from rich.text import Text

console = Console()

REF_START_VIZ: float = 0.0
MAX_CONCURRENT_VIZ = 3
N_TASKS_VIZ = 12


async def fake_llm_timed(task_id: int, semaphore: asyncio.Semaphore) -> dict:
    global REF_START_VIZ
    created = time.perf_counter() - REF_START_VIZ
    async with semaphore:
        acquired = time.perf_counter() - REF_START_VIZ
        duration = random.uniform(0.8, 2.5)
        await asyncio.sleep(duration)
        released = time.perf_counter() - REF_START_VIZ
    return {"id": task_id, "created": created, "acquired": acquired, "released": released}


def make_gantt_bar(created: float, acquired: float, released: float, total_time: float, width: int = 60) -> Text:
    """Create a text Gantt bar: . = waiting, # = active, space = idle."""
    bar = [" "] * width
    scale = width / total_time

    wait_start = int(created * scale)
    wait_end = int(acquired * scale)
    active_end = int(released * scale)

    for i in range(min(wait_start, width), min(wait_end, width)):
        bar[i] = "."
    for i in range(min(wait_end, width), min(active_end, width)):
        bar[i] = "#"

    text = Text()
    for ch in bar:
        if ch == ".":
            text.append(ch, style="yellow")
        elif ch == "#":
            text.append(ch, style="green")
        else:
            text.append(ch)
    return text


async def demo_gantt():
    global REF_START_VIZ
    REF_START_VIZ = time.perf_counter()

    semaphore = asyncio.Semaphore(MAX_CONCURRENT_VIZ)
    results = await asyncio.gather(
        *[fake_llm_timed(i, semaphore) for i in range(N_TASKS_VIZ)]
    )
    results.sort(key=lambda x: x["acquired"])

    total_time = max(r["released"] for r in results)

    console.print(f"[bold cyan]Gantt Chart — Semaforo({MAX_CONCURRENT_VIZ}), {N_TASKS_VIZ} tasks[/bold cyan]")
    console.print(f"  [yellow].[/yellow] = aguardando slot   [green]#[/green] = executando   ' ' = inativo\n")

    table = Table(show_header=True, box=box.SIMPLE, show_lines=False)
    table.add_column("Task", width=8, style="bold")
    table.add_column("Espera", width=8, justify="right")
    table.add_column("Ativo", width=8, justify="right")
    table.add_column("Timeline (cada char ~ escala de tempo)", min_width=62)

    for r in results:
        wait = r["acquired"] - r["created"]
        active = r["released"] - r["acquired"]
        bar = make_gantt_bar(r["created"], r["acquired"], r["released"], total_time)
        table.add_row(
            f"Task-{r['id']:02d}",
            f"{wait:.2f}s",
            f"{active:.2f}s",
            bar,
        )

    console.print(table)
    console.print(f"[dim]Tempo total: {total_time:.2f}s[/dim]")


await demo_gantt()

## 5. Na pratica: AgentSemaphore

O `asyncio.Semaphore` basico tem uma limitacao: ele deixa tarefas esperando indefinidamente. Em um sistema de producao, precisamos de:

1. **Limite de concorrencia** — max N tarefas simultaneas
2. **Fila com capacidade maxima** — max M tarefas aguardando
3. **Rejeicao rapida** — se fila cheia, recusar imediatamente (fail-fast)
4. **Metricas** — quantas foram aceitas, enfileiradas, rejeitadas

Esse e exatamente o papel do `AgentSemaphore` no modulo `concurrency.py` do projeto.

In [ ]:
import asyncio
import random
import time
from dataclasses import dataclass, field
from enum import Enum

from rich.console import Console
from rich.table import Table
from rich import box
from rich.panel import Panel

console = Console()


class RequestStatus(Enum):
    ACCEPTED = "accepted"   # acquired semaphore immediately
    QUEUED = "queued"       # waited in queue then ran
    REJECTED = "rejected"   # queue was full, request dropped


@dataclass
class AgentSemaphore:
    """Semaphore with bounded queue and rejection for LLM agent workloads."""

    max_concurrent: int = 3
    max_queue: int = 5

    _semaphore: asyncio.Semaphore = field(init=False)
    _queue_count: int = field(init=False, default=0)
    _stats: dict = field(init=False)

    def __post_init__(self):
        self._semaphore = asyncio.Semaphore(self.max_concurrent)
        self._stats = {"accepted": 0, "queued": 0, "rejected": 0, "completed": 0}

    async def run(self, coro, task_id: int) -> dict:
        """
        Attempt to run a coroutine with semaphore control.

        - If a slot is immediately free: run right away (ACCEPTED)
        - If slots are full but queue has space: wait in queue (QUEUED)
        - If queue is also full: reject immediately (REJECTED)
        """
        created_at = time.perf_counter()

        # Check if we can even queue this request
        if self._semaphore._value == 0:  # all slots busy
            if self._queue_count >= self.max_queue:
                self._stats["rejected"] += 1
                return {"id": task_id, "status": RequestStatus.REJECTED, "wait": 0, "duration": 0}
            # Will queue
            self._queue_count += 1
            status = RequestStatus.QUEUED
            self._stats["queued"] += 1
        else:
            status = RequestStatus.ACCEPTED
            self._stats["accepted"] += 1

        async with self._semaphore:
            if status == RequestStatus.QUEUED:
                self._queue_count -= 1
            acquired_at = time.perf_counter()
            wait_time = acquired_at - created_at

            result = await coro
            duration = time.perf_counter() - acquired_at

        self._stats["completed"] += 1
        return {
            "id": task_id,
            "status": status,
            "wait": wait_time,
            "duration": duration,
        }

    @property
    def stats(self) -> dict:
        return dict(self._stats)


async def simulate_work(task_id: int) -> str:
    """Simulate LLM processing time."""
    await asyncio.sleep(random.uniform(1.0, 2.5))
    return f"result-{task_id}"


async def demo_agent_semaphore():
    agent_sem = AgentSemaphore(max_concurrent=3, max_queue=5)
    n_requests = 15

    console.print(
        Panel(
            f"[bold]AgentSemaphore[/bold]\n"
            f"  max_concurrent = {agent_sem.max_concurrent}\n"
            f"  max_queue      = {agent_sem.max_queue}\n"
            f"  requests       = {n_requests}\n\n"
            f"  Capacidade total: {agent_sem.max_concurrent + agent_sem.max_queue} requests\n"
            f"  Esperados rejeitados: {max(0, n_requests - agent_sem.max_concurrent - agent_sem.max_queue)}",
            title="Configuracao",
            border_style="cyan",
        )
    )

    results = await asyncio.gather(
        *[
            agent_sem.run(simulate_work(i), task_id=i)
            for i in range(n_requests)
        ]
    )

    table = Table(title="Resultado por Request", show_lines=True, box=box.ROUNDED)
    table.add_column("Request", style="bold", width=10)
    table.add_column("Status", width=12)
    table.add_column("Espera", justify="right", width=10)
    table.add_column("Duracao", justify="right", width=10)

    status_style = {
        RequestStatus.ACCEPTED: "green",
        RequestStatus.QUEUED: "yellow",
        RequestStatus.REJECTED: "red",
    }

    for r in sorted(results, key=lambda x: x["id"]):
        st = r["status"]
        style = status_style[st]
        table.add_row(
            f"Req-{r['id']:02d}",
            f"[{style}]{st.value.upper()}[/{style}]",
            f"{r['wait']:.2f}s" if st != RequestStatus.REJECTED else "—",
            f"{r['duration']:.2f}s" if st != RequestStatus.REJECTED else "—",
        )

    console.print(table)

    stats = agent_sem.stats
    console.print(Panel(
        f"  [green]Aceitas imediatamente:[/green]  {stats['accepted']}\n"
        f"  [yellow]Enfileiradas (esperaram):[/yellow] {stats['queued']}\n"
        f"  [red]Rejeitadas (fila cheia):[/red]  {stats['rejected']}\n"
        f"  [cyan]Completadas com sucesso:[/cyan] {stats['completed']}",
        title="Estatísticas Finais",
        border_style="green",
    ))


await demo_agent_semaphore()

## Tente Voce

### Exercicio 1 — Ajuste os slots
No codigo da celula anterior, mude `max_concurrent=3` para `max_concurrent=5` e `max_queue=3`.  
O que muda no numero de rejeitados? Por que?

### Exercicio 2 — Implemente prioridade
O `AgentSemaphore` atual trata todos os requests igualmente (FIFO). Modifique-o para aceitar um parametro `priority: int` no metodo `run()`. Requests de prioridade alta (priority=1) devem ser processados antes de prioridade baixa (priority=2).  
Dica: use `asyncio.PriorityQueue` internamente.

### Exercicio 3 — Timeout na fila
Adicione um parametro `queue_timeout: float` ao `AgentSemaphore`. Se uma task passar mais que `queue_timeout` segundos aguardando na fila, ela deve ser cancelada com status `TIMEOUT` (sem precisar esperar o slot).  
Dica: use `asyncio.wait_for()` com `asyncio.shield()` para separar o timeout de espera do timeout de execucao.